# Getting started: single-FOV SMLM fitting

This is the same worked example shown in the docs (`docs/getting-started.rst`) and the
project `README`, run end-to-end here against data that ships with the repository — no
external data or path edits needed.

It covers the four core `AnalysisPipeline` steps:

1. **Calibration** — load a pre-computed camera calibration (gain/offset/variance/read
   noise/relative QE maps).
2. **Fitting** — detect and fit single-molecule PSFs in a Bayer-mosaic camera stack.
3. **Loading localisations** — read the fitted results back from disk.
4. **Filtering & clustering** — apply quality filters and group repeat localisations of
   the same molecule into a per-molecule table.

The data is `test_tiffs/single_FOV_smlm/` — a small, synthetic single-dye (ATTO 647N)
acquisition simulated with a known ground truth (see its `README.md` for details) via
`pyS3M.simulation.pattern_source.simulate_acquisition` — the same backend
`notebooks/simulations/01_simulating_an_acquisition.ipynb` walks through directly.

## Setup

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt

from pyS3M.AnalysisPipeline import AnalysisPipeline, FittingConfig
from pyS3M.Constants import AnalysisConfig
import pyS3M.PlottingBase as PlottingBase

REPO_ROOT = Path.cwd().resolve().parents[1]  # notebooks/analyses/ -> repo root
DATA_DIR = REPO_ROOT / "test_tiffs" / "single_FOV_smlm"
CALIBRATION_DIR = REPO_ROOT / "Camera_Calibrations" / "Ximea_Camera"

plotter = PlottingBase.PublicationPlotter()

print(DATA_DIR, DATA_DIR.is_dir())
print(CALIBRATION_DIR, CALIBRATION_DIR.is_dir())

## 1. Load calibration

Camera calibration (gain/offset/variance/read-noise/relative-QE maps) converts raw ADU
counts into calibrated photoelectron statistics that the fitter needs. `Camera_Calibrations/`
ships pre-computed maps for both supported cameras, so `load_calibration` is all that's
needed here.

If you have your own raw flat-field/dark frames instead, `AnalysisPipeline.calibrate(raw_dir)`
computes a fresh calibration from them (also available from the GUI's "CMOS Calibration"
tab, which documents the required raw-data folder layout) — not run here, since no raw
calibration exemplar dataset ships with the repo.

In [ ]:
cfg = AnalysisConfig(display=False)
pipe = AnalysisPipeline(camera="ximea", config=cfg)
pipe.load_calibration(CALIBRATION_DIR)
print("Calibration loaded:", pipe.gain_map.shape)

## 2. Fit

`FittingConfig.peak_wavelength` should match the simulated/labelled dye's emission —
`test_tiffs/single_FOV_smlm/README.md` states the recommended value (0.685 µm) for this
fixture.

In [ ]:
fc = FittingConfig(peak_wavelength=0.685, pfa=1e-3)
pipe.fit(DATA_DIR, mode="smlm", fitting_config=fc)

## 3. Load localisations

In [ ]:
locs = pipe.load_localisations(DATA_DIR)
print(f"{len(locs)} localisations")
locs.head()

## 4. Filter & cluster

`filter_and_cluster` applies quality-filtering thresholds (`FilteringCriteria`) and then
clusters repeat localisations of the same molecule (`ClusteringConfig`, HDBSCAN by
default), returning two tables:

- `sm_db` — one row per **molecule** (clustered, higher-SNR average).
- `sf_db` — one row per **localisation** (single-frame, pre-averaging), carrying a
  `molecular_index` column linking it back to its parent row in `sm_db`.

**`min_cluster_size` needs to match the imaging modality.** `single_FOV_smlm` is a
**STORM**-style fixture: each labelled site photobleaches permanently after a handful of
on/off cycles (5, here), so no single molecule is ever seen more than a few times. HDBSCAN's
default `min_cluster_size=10` is tuned for **PAINT**-style repeat-visit blinking instead
(continuously-exchanging transient probes revisit the same site many times — see
`04_clustering_and_quality_filtering.ipynb`, which uses a PAINT fixture and the default
cluster size). Passing the PAINT-sized default here would find no clusters at all — every
localisation is legitimately clustered into groups of one to a few, so `min_cluster_size`
must be small enough to allow that.

In [ ]:
from pyS3M.clustering import ClusteringConfig

cc = ClusteringConfig(min_cluster_size=2)
sm_db, sf_db = pipe.filter_and_cluster(locs, clustering_config=cc)
print(f"{len(sm_db)} molecules from {len(sf_db)} quality-filtered localisations")

## Render

In [ ]:
fig, ax = plotter.create_figure()
pixel_size_nm = pipe.pixel_size * 1000.0
plotter.scatter_plot(ax, sm_db["xc"] * pixel_size_nm, sm_db["yc"] * pixel_size_nm,
                      xlabel="x (nm)", ylabel="y (nm)", size=8, alpha=0.8)
ax.set_aspect("equal")
ax.set_title(f"{len(sm_db)} molecules — single_FOV_smlm")
plt.show()

## Next steps

- `02_multi_fov_fitting.ipynb` — fitting several fields of view in one folder.
- `03_drift_correction.ipynb` — AIM drift correction, including a real segmentation/`roi_r`
  trade-off this fixture is specifically built to demonstrate.
- `04_clustering_and_quality_filtering.ipynb` — a closer look at `FilteringCriteria` and
  `ClusteringConfig`.
- `05_channel_unmixing.ipynb` — separating spectrally-distinct dye populations.
- `06_nile_red_pixelated_pixel3_qe.ipynb` — Nile Red environment-sensing analysis (pixelated relative pixel 3 QE).
- `07_frc.ipynb` — estimating spatial resolution (Fourier Ring Correlation).
- `../simulations/` — how the `test_tiffs/` fixtures used throughout these notebooks were
  generated, and how to simulate your own.